In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
labels = pd.read_csv('/kaggle/input/richters-prediction-dataset/train_labels.csv')
values = pd.read_csv('/kaggle/input/richters-prediction-dataset/train_values.csv')
print('datasets loaded')

datasets loaded


In [3]:
labels.head(3)

,building_id,damage_grade
0,802906,3
1,28830,2
2,94947,3


In [4]:
values.head(3)

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other
0,802906,6,487,12198,2,30,6,5,t,r,...,0,0,0,0,0,0,0,0,0,0
1,28830,8,900,2812,2,10,8,7,o,r,...,0,0,0,0,0,0,0,0,0,0
2,94947,21,363,8973,2,10,5,5,t,r,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# merge both datasets
df = pd.merge(labels,values, on='building_id', how='outer')
print("merge successful.")

merge successful.


In [7]:
# view merged dataset
df.head()

,building_id,damage_grade,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,...,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other
0,4,2,30,266,1224,1,25,5,2,t,...,0,0,0,0,0,0,0,0,0,0
1,8,3,17,409,12182,2,0,13,7,t,...,0,0,0,0,0,0,0,0,0,0
2,12,3,17,716,7056,2,5,12,6,o,...,0,0,0,0,0,0,0,0,0,0
3,16,2,4,651,105,2,80,5,4,n,...,0,0,0,0,0,0,0,0,0,0
4,17,2,3,1387,3909,5,40,5,10,t,...,0,0,0,0,0,0,0,0,0,0


In [4]:
total_null_values = df.isnull().sum().sum()
print('total null values = ', total_null_values)

total null values =  0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260601 entries, 0 to 260600
Data columns (total 40 columns):
 #   Column                                  Non-Null Count   Dtype 
---  ------                                  --------------   ----- 
 0   building_id                             260601 non-null  int64 
 1   damage_grade                            260601 non-null  int64 
 2   geo_level_1_id                          260601 non-null  int64 
 3   geo_level_2_id                          260601 non-null  int64 
 4   geo_level_3_id                          260601 non-null  int64 
 5   count_floors_pre_eq                     260601 non-null  int64 
 6   age                                     260601 non-null  int64 
 7   area_percentage                         260601 non-null  int64 
 8   height_percentage                       260601 non-null  int64 
 9   land_surface_condition                  260601 non-null  object
 10  foundation_type                         260601 non-null 

In [5]:
X = df.drop(columns=['building_id','damage_grade'], axis=1)
y = df['damage_grade']

In [7]:
# create list of individual data types
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

In [8]:
print('cat cols = \n', cat_cols)
print("\n total : ", len(cat_cols))

cat cols = 
 Index(['land_surface_condition', 'foundation_type', 'roof_type',
       'ground_floor_type', 'other_floor_type', 'position',
       'plan_configuration', 'legal_ownership_status'],
      dtype='object')

 total :  8


In [9]:
print('num cols = \n', num_cols)
print("\n total : ", len(num_cols))

num cols = 
 Index(['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id',
       'count_floors_pre_eq', 'age', 'area_percentage', 'height_percentage',
       'has_superstructure_adobe_mud', 'has_superstructure_mud_mortar_stone',
       'has_superstructure_stone_flag',
       'has_superstructure_cement_mortar_stone',
       'has_superstructure_mud_mortar_brick',
       'has_superstructure_cement_mortar_brick', 'has_superstructure_timber',
       'has_superstructure_bamboo', 'has_superstructure_rc_non_engineered',
       'has_superstructure_rc_engineered', 'has_superstructure_other',
       'count_families', 'has_secondary_use', 'has_secondary_use_agriculture',
       'has_secondary_use_hotel', 'has_secondary_use_rental',
       'has_secondary_use_institution', 'has_secondary_use_school',
       'has_secondary_use_industry', 'has_secondary_use_health_post',
       'has_secondary_use_gov_office', 'has_secondary_use_use_police',
       'has_secondary_use_other'],
      dtype='object')

 to

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        # encoding for categorical features
        ('cat', OneHotEncoder(handle_unknown='ignore',sparse_output=False), cat_cols),
        # no normalization to numerical values
        ('num', 'passthrough', num_cols)
    ],
    verbose=True,
    n_jobs=-1,
    remainder='passthrough'
)

In [41]:
# create model final pipeline
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)
print("dataset split successful.")

dataset split successful.


In [43]:
# hyperparameter tuning with GridSearchCV on RandomforestClassifier.
# dictionary of hyperparameters
param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=2
)

print("fitting...")
grid.fit(X_train, y_train)
print('fitting completed')

fitting...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END model__max_depth=None, model__min_samples_split=2, model__n_estimators=200; total time= 1.3min
[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=200; total time= 1.2min
[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=200; total time= 1.2min


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END model__max_depth=None, model__min_samples_split=2, model__n_estimators=200; total time= 1.3min
[CV] END model__max_depth=None, model__min_samples_split=2, model__n_estimators=400; total time= 2.5min
[CV] END model__max_depth=None, model__min_samples_split=2, model__n_estimators=400; total time= 2.5min
[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=200; total time= 1.2min
[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=400; total time= 2.3min
[CV] END model__max_depth=10, model__min_samples_split=2, model__n_estimators=400; total time= 1.2min
[CV] END model__max_depth=10, model__min_samples_split=5, model__n_estimators=400; total time= 1.2min
[CV] END model__max_depth=20, model__min_samples_split=2, model__n_estimators=200; total time=  54.8s
[CV] END model__max_depth=10, model__min_samples_split=2, model__n_estimators=200; total time=  36.9s
[CV] END model__max_depth=10, model__min_samples_split=2, model__n_estim

In [11]:
# function to evaluate model
def evaluate_model(model, X_val, y_val):
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='micro')
    return acc,f1

In [54]:
best_rf_model = grid.best_estimator_
acc_rf, f1_rf = evaluate_model(best_rf_model, X_val, y_val)
print(f"accuracy = {acc_rf:.4f}")
print(f"F1 score = {f1_rf}")
print("\nClassification Report:\n")
print(classification_report(y_val, best_rf_model.predict(X_val)))

accuracy = 0.7236
F1 score = 0.7236113029730933

Classification Report:

              precision    recall  f1-score   support

           1       0.68      0.46      0.55      6281
           2       0.72      0.85      0.78     37065
           3       0.74      0.59      0.66     21805

    accuracy                           0.72     65151
   macro avg       0.71      0.63      0.66     65151
weighted avg       0.72      0.72      0.72     65151



In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)


pipeline_lgb = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', lgb_model)
])

param_grid_lgb = {
    'model__n_estimators': [300, 600],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [10, 20],
    'model__num_leaves': [31, 63],
}
grid_lgb = GridSearchCV(
    pipeline_lgb,
    param_grid_lgb,
    scoring='f1_macro',   
    cv=3,
    n_jobs=-1,
    verbose=2
)

print("fitting started")
grid_lgb.fit(X_train, y_train)
print("fitting completed")

fitting started
Fitting 3 folds for each of 16 candidates, totalling 48 fits


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [17]:
best_lgb = grid_lgb.best_estimator_
best_acc, best_f1 = evaluate_model(best_lgb, X_val, y_val)
print(f"accuracy = {best_acc:.4f}")
print(f"F1 score = {grid_f1}")
print("\nClassification Report:\n")
print(classification_report(y_val, best_lgb.predict(X_val)))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


accuracy = 0.6423
F1 score = 0.6423385673282068

Classification Report:



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           1       0.40      0.82      0.54      6281
           2       0.77      0.56      0.65     37065
           3       0.62      0.73      0.67     21805

    accuracy                           0.64     65151
   macro avg       0.60      0.70      0.62     65151
weighted avg       0.69      0.64      0.65     65151



In [18]:
# convert output {1,2,3} into {0,1,2}
xgb_y_train = y_train - 1
xgb_y_val = y_val - 1

In [22]:
# XGBoost model
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=7,
            objective='multi:softprob',
            random_state=42
        )

xgb_pipe = Pipeline(
    steps=[
        ('preprocessing', preprocessor),
        ('model', xgb_model)
    ])

print("fitting started")
xgb_pipe.fit(X_train, xgb_y_train)
print("fitting completed")

y_pred_xgb = xgb_pipe.predict(X_val) + 1
print("sccuracy = ", accuracy_score(y_val, y_pred_xgb))
print("f1 score (mirco) = ", f1_score(y_val, y_pred_xgb, average='micro'))
print("classification report = \n", classification_report(y_val, y_pred_xgb))

fitting started
fitting completed
sccuracy =  0.7309788030882105
f1 score (mirco) =  0.7309788030882105
classification report = 
               precision    recall  f1-score   support

           1       0.70      0.47      0.56      6281
           2       0.73      0.85      0.78     37065
           3       0.75      0.60      0.67     21805

    accuracy                           0.73     65151
   macro avg       0.73      0.64      0.67     65151
weighted avg       0.73      0.73      0.72     65151



In [23]:
# load test dataset
test_df = pd.read_csv(r'/kaggle/input/earthquake-severity-test-dataset/test_values.csv')
print(test_df.head(3))

   building_id  geo_level_1_id  geo_level_2_id  geo_level_3_id  \
0       300051              17             596           11307   
1        99355               6             141           11987   
2       890251              22              19           10044   

   count_floors_pre_eq  age  area_percentage  height_percentage  \
0                    3   20                7                  6   
1                    2   25               13                  5   
2                    2    5                4                  5   

  land_surface_condition foundation_type  ... has_secondary_use_agriculture  \
0                      t               r  ...                             0   
1                      t               r  ...                             1   
2                      t               r  ...                             0   

  has_secondary_use_hotel has_secondary_use_rental  \
0                       0                        0   
1                       0                

In [31]:
print(test_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86868 entries, 0 to 86867
Data columns (total 39 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   building_id                             86868 non-null  int64 
 1   geo_level_1_id                          86868 non-null  int64 
 2   geo_level_2_id                          86868 non-null  int64 
 3   geo_level_3_id                          86868 non-null  int64 
 4   count_floors_pre_eq                     86868 non-null  int64 
 5   age                                     86868 non-null  int64 
 6   area_percentage                         86868 non-null  int64 
 7   height_percentage                       86868 non-null  int64 
 8   land_surface_condition                  86868 non-null  object
 9   foundation_type                         86868 non-null  object
 10  roof_type                               86868 non-null  object
 11  gr

In [24]:
# check null values:
print("total null values = ", test_df.isnull().sum())

total null values =  building_id                               0
geo_level_1_id                            0
geo_level_2_id                            0
geo_level_3_id                            0
count_floors_pre_eq                       0
age                                       0
area_percentage                           0
height_percentage                         0
land_surface_condition                    0
foundation_type                           0
roof_type                                 0
ground_floor_type                         0
other_floor_type                          0
position                                  0
plan_configuration                        0
has_superstructure_adobe_mud              0
has_superstructure_mud_mortar_stone       0
has_superstructure_stone_flag             0
has_superstructure_cement_mortar_stone    0
has_superstructure_mud_mortar_brick       0
has_superstructure_cement_mortar_brick    0
has_superstructure_timber                 0
has_superst

In [25]:
# Always keep building_id for submission
test_ids = test_df["building_id"]

# Drop id before prediction
X_test = test_df.drop(columns=["building_id"])

In [48]:
# Predict using best trained pipeline
test_predictions = best_rf_model.predict(X_test)

[CV] END model__max_depth=20, model__min_samples_split=2, model__n_estimators=400; total time= 1.8min
[CV] END model__max_depth=20, model__min_samples_split=5, model__n_estimators=400; total time= 1.2min


In [26]:
# cliassify values using XGBclassifier
test_predictions = xgb_pipe.predict(X_test)

In [28]:
# function to save submission file
def submit(predictions, ids, file_name):
    submission = pd.DataFrame({
        "building_id": ids,
        "damage_grade": predictions
    })
    submission.to_csv(file_name, index=False)
    print(f"{file_name} saved.")

In [51]:
# execute function (create submission file)
submit(predictions=test_predictions, ids=test_ids, file_name='submission_1.csv')

submission_1.csv saved.


In [29]:
# save submission for XGB model
submit(predictions=test_predictions, ids=test_ids, file_name='submission_2.csv')

submission_2.csv saved.
[ColumnTransformer] ........... (2 of 2) Processing num, total=   0.0s
[ColumnTransformer] ........... (1 of 2) Processing cat, total=   0.4s
